In [1]:
!pip install -q -U "transformers[serving]" accelerate bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
import os, subprocess, time
# os.environ["HF_TOKEN"] = "hf_..."   # only for gated models
subprocess.Popen(
    ["transformers", "serve", "--port", "8000",
     "--quantization", "bnb-4bit",      # remove for fp16
     "--model-timeout", "3600"],        # idle seconds before unload
    stdout=open("serve.log", "w"), stderr=subprocess.STDOUT)
time.sleep(15)
print(open("serve.log").read()[-500:])

In [3]:
import re, time, subprocess
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stderr=open("cf.log", "w"),
)
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        URL = m.group(0)
        print("BASE URL:", URL + "/v1")
        break

BASE URL: https://tumor-correlation-chest-barn.trycloudflare.com/v1


In [4]:
!pip install -q fastapi uvicorn sentence-transformers

In [5]:
%%writefile embed_server.py
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

app = FastAPI()
m = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
m.max_seq_length = 256

class Req(BaseModel):
    model: str | None = None
    input: str | list[str]

@app.post("/v1/embeddings")
def emb(r: Req):
    xs = [r.input] if isinstance(r.input, str) else r.input
    v = m.encode(xs, normalize_embeddings=True, batch_size=16)
    return {"object": "list", "model": "bge-small-en-v1.5",
            "data": [{"object": "embedding", "index": i, "embedding": e.tolist()} for i, e in enumerate(v)],
            "usage": {"prompt_tokens": 0, "total_tokens": 0}}

Writing embed_server.py


In [6]:
subprocess.Popen(["uvicorn", "embed_server:app", "--port", "8001"],
                 stdout=open("embed.log", "w"), stderr=subprocess.STDOUT)
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8001"],
                 stderr=open("cf2.log", "w"))
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf2.log").read())
    if m:
        print("EMBED URL:", m.group(0) + "/v1")
        break

EMBED URL: https://late-power-essential-soccer.trycloudflare.com/v1
